# セミナーII 第1回 実践：画像分類モデルを学習する

スライドの「画像分類の実践」で見た学習を、実際に動かします。題材は、犬と猫の写真を37クラスに分類する問題です（Oxford-IIIT Pet）。

**最初に**、メニューの「ランタイム」→「ランタイムのタイプを変更」で **GPU**（T4 など）を選んでから、上のセルから順に実行してください。

コードを全部書ける必要はありません。スライドのどの話が、コードのどこにあるかを説明できることを目標にします。各節の見出しに、対応するスライドのページを書いています。

## 0. このノートブックの地図

### 学習に用意するもの（p82）と、この後の節

| 用意するもの | スライド | このノートブック |
| --- | --- | --- |
| 計算環境 | GPU を使う理由（p95） | 1. 準備 |
| 学習の設定 | 人間が決める学習の設定（p93） | 2. 学習の設定 |
| データセット | ① データを用意する（p84〜p88） | 3. ① データを用意する |
| モデル $F$ | CNN による画像分類（p73）、学習済みモデルから始める（p94） | 4. モデル F とパラメータ θ |
| Loss | 分類の Loss：交差エントロピー（p75） | 5. Loss と更新の方法 |
| 更新の方法 | 勾配降下法（p25）、CNN の学習も、同じ繰り返し（p76） | 5. Loss と更新の方法 |

そろったら、**6. ② 学習を進める**（p90〜p92）で学習し、**7. ③ 結果を確かめる**（p97〜p99）、**8. 推論**（p17）と進みます。**9.** では、同じ学習を GitHub でよく見る形（`train.py`）で動かします。

### コンセプト式とコードの対応（p6、p16）

$$Y = F(X; \theta)$$

| 式 | 意味 | コード |
| --- | --- | --- |
| $X$ | 入力（画像） | `images` |
| $Y$ | 正解（クラスの番号） | `labels` |
| $F$ | 計算の形（ネットワークの形） | `model` の構造（ResNet-18） |
| $\theta$ | パラメータ（重み） | `model.parameters()` |
| $\hat{Y}$ | 予測 | `logits = model(images)` |
| Loss | 予測と正解のずれ | `loss = criterion(logits, labels)` |

## 1. 準備（計算環境。p95）

公開リポジトリから、このノートブックが使うファイル（`.py` と画像）を取得します。取得したファイルは、左の「ファイル」（フォルダのアイコン）から開けます。

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/OWNER/seminar2-2026.git"  # ★ 公開後に記入する
REPO_REF = "main"  # 授業当日は、教材が変わらないようタグを指定する

if not Path("viz.py").exists():  # Colab では、最初はファイルが無い
    if not Path("seminar2-2026").exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, "seminar2-2026"],
                       check=True)
    os.chdir("seminar2-2026/lesson01")
print("作業フォルダ:", os.getcwd())
print(sorted(p.name for p in Path(".").iterdir()))

In [ ]:
# Colab に入っていないライブラリがあれば入れる
import importlib.util

for package in ["timm", "datasets"]:
    if importlib.util.find_spec(package) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

In [ ]:
# .py を書き換えたら、実行し直さなくても反映されるようにする
%load_ext autoreload
%autoreload 2

import copy
import json
import time
from functools import partial

import numpy as np
import timm
import torch
import torch.nn as nn
from datasets import load_dataset
from PIL import Image
from timm.data import IMAGENET_DEFAULT_MEAN, IMAGENET_DEFAULT_STD
from torch.utils.data import DataLoader
from torchvision import transforms

import viz  # 表示用の関数（viz.py）。学習の仕組みとは関係しない

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, torch.cuda.get_device_name(0) if device == "cuda" else "（GPU を選ぶと速くなります）")

## 2. 学習の設定（p93）

$\theta$ は機械が求めますが、学習の設定（ハイパーパラメータ）は人間が決めます。実験で変えるのは、基本的にこのセルだけです。

In [ ]:
MODEL_NAME = "resnet18.a1_in1k"  # モデル F（timm の名前）
PRETRAINED = True                # 学習済みの重みから始めるか（p94）
EPOCHS = 3                       # データ全体を何周するか（p91）
BATCH_SIZE = 64                  # 1回の更新に使う枚数（p92）
LEARNING_RATE = 1e-3             # 学習率（p26）
SEED = 0                         # 乱数の種。同じなら、分け方や初期値が同じになる

IMAGE_DIR = None  # 自分で集めた画像で学習するときは、フォルダ名を書く（10. 課題の発展）

# データセットの取得元。授業の途中で内容が変わらないよう、リビジョンを固定している
DATASET_ID = "timm/oxford-iiit-pet"
DATASET_REVISION = "089695c834a7deb60505b7cc506672db1c31a6aa"

# GPU が無いときは、枚数を減らして早く終わらせる（None は全部）
MAX_TRAIN_IMAGES = None if device == "cuda" else 600
MAX_TEST_IMAGES = None if device == "cuda" else 300
NUM_WORKERS = 2 if device == "cuda" and os.name != "nt" else 0  # データを読む並列数

torch.manual_seed(SEED)

## 3. ① データを用意する（p84〜p88）

### 3-1. データセット：Oxford-IIIT Pet（p84）

犬と猫の写真に、品種名の正解ラベルが付いたデータセットです。Hugging Face Hub から取得します（約800MB、初回のみ）。ログインは不要で、`HF_TOKEN` に関する警告が表示されても無視してかまいません。

In [ ]:
if IMAGE_DIR is None:
    dataset = load_dataset(DATASET_ID, revision=DATASET_REVISION)
else:
    # フォルダ名がクラス名になる。test が無いので、2割を test として取り分ける
    images = load_dataset("imagefolder", data_dir=IMAGE_DIR)["train"]
    dataset = images.train_test_split(test_size=0.2, seed=SEED, stratify_by_column="label")
print(dataset)

class_names = dataset["train"].features["label"].names
num_classes = len(class_names)
print("クラスの数:", num_classes)
print(class_names)

In [ ]:
# スライドと同じ8クラスの見本。括弧の中はクラスの番号
viz.show_samples(dataset["train"], class_names,
                 ["Persian", "Siamese", "Sphynx", "British_Shorthair", "shiba_inu", "pug", "beagle", "samoyed"])

### 3-2. データ1件：画像と正解ラベルの組（p85）

データセットの1件は、入力の画像と、正解のラベル（クラスの番号）の組です（p15）。クラス名は 0〜36 の番号で扱います。

In [ ]:
sample = dataset["train"][49]  # スライドと同じ柴犬の写真
print(sample.keys())
print("正解ラベル:", sample["label"], "→", class_names[sample["label"]])
viz.show_example(sample, class_names)

In [ ]:
# 画像は数の並び（p48）。高さ × 幅 × 3（赤、緑、青）の、0〜255 の整数
pixels = np.array(sample["image"].convert("RGB"))
print("形:", pixels.shape, " 型:", pixels.dtype)
h, w = pixels.shape[:2]
print("中央の画素の (赤, 緑, 青):", pixels[h // 2, w // 2])

### 3-3. データを3つに分ける（p86）

学習に使っていないデータで評価するために、データを3つに分けます。Oxford-IIIT Pet には train と test しかないので、train の2割を validation に回します。

| データ | 役割 | たとえ |
| --- | --- | --- |
| train | 重みの更新に使う | 練習問題 |
| validation | 設定の判断、重みの選択 | 模試 |
| test | 最後に1回だけ測る | 本番の試験 |

In [ ]:
# 各クラスの比率がそろうように分ける（stratify）
split = dataset["train"].train_test_split(test_size=0.2, seed=SEED, stratify_by_column="label")
train_data, val_data, test_data = split["train"], split["test"], dataset["test"]

if MAX_TRAIN_IMAGES is not None:
    train_data = train_data.shuffle(seed=SEED).select(range(min(MAX_TRAIN_IMAGES, len(train_data))))
    val_data = val_data.shuffle(seed=SEED).select(range(min(MAX_TRAIN_IMAGES // 4, len(val_data))))
if MAX_TEST_IMAGES is not None:
    test_data = test_data.shuffle(seed=SEED).select(range(min(MAX_TEST_IMAGES, len(test_data))))

print("train      :", len(train_data))
print("validation :", len(val_data))
print("test       :", len(test_data))

### 3-4. 前処理：大きさと値の範囲をそろえる（p87）

1. **大きさをそろえる**：縮小して、中央を 224×224 で切り抜く
2. **値の範囲をそろえる（正規化）**：値 ＝（画素値 ÷ 255 − 平均）÷ 標準偏差。チャネルごとに計算する

学習済みモデルを使うときは、そのモデルを学習したとき（ImageNet）と同じ前処理にします。下の関数は、学習用（3-5 のデータ拡張つき）と評価用の2つを作ります。

In [ ]:
IMAGE_SIZE = 224
# 学習済みモデルを ImageNet で学習したときの平均と標準偏差（p87）
MEAN = IMAGENET_DEFAULT_MEAN
STD = IMAGENET_DEFAULT_STD


def build_transforms(image_size=IMAGE_SIZE, mean=MEAN, std=STD):
    """前処理（p87）とデータ拡張（p88）を作る。学習用と評価用の2つを返す"""
    normalize = transforms.Normalize(mean, std)
    # 評価用：大きさをそろえて（縮小して中央を切り抜く）、値の範囲をそろえる
    eval_transform = transforms.Compose([
        transforms.Resize(int(image_size * 1.15)),
        transforms.CenterCrop(image_size),
        transforms.ToTensor(),
        normalize,
    ])
    # 学習用：毎回ランダムに一部を切り抜いて拡大し、左右を反転する
    train_transform = transforms.Compose([
        transforms.RandomResizedCrop(image_size, scale=(0.5, 1.0)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        normalize,
    ])
    return train_transform, eval_transform


train_transform, eval_transform = build_transforms()
eval_transform

In [ ]:
print("平均      (赤, 緑, 青):", MEAN)
print("標準偏差  (赤, 緑, 青):", STD)
x = eval_transform(sample["image"])
print("前処理の後の形 (チャネル, 高さ, 幅):", tuple(x.shape))
print("値の範囲:", round(x.min().item(), 2), "〜", round(x.max().item(), 2))

# スライドの数値例：赤の値が 191 の画素
print("(191 ÷ 255 − 0.485) ÷ 0.229 =", round((191 / 255 - MEAN[0]) / STD[0], 2))

viz.show_preprocess(sample["image"], eval_transform)

### 3-5. データ拡張：見え方を変えて増やす（p88）

学習のときだけ、毎回ランダムに「一部を切り抜いて拡大（面積の50〜100%）」「左右を反転」します。同じ画像を丸覚えしにくくなります。評価（validation、test）では使いません。

下のセルを何回か実行すると、毎回違う画像になることが分かります。

In [ ]:
train_transform

In [ ]:
viz.show_augmentations(sample["image"], train_transform)

### 3-6. mini-batch：少しずつ取り出す（p92）

`DataLoader` は、画像に前処理をかけてテンソルにし、`BATCH_SIZE` 枚ずつ束ねて取り出します。train は毎 epoch、順番を混ぜます（`shuffle=True`）。

In [ ]:
def _apply(batch, transform):
    return {
        "image": [transform(img.convert("RGB")) for img in batch["image"]],
        "label": batch["label"],
    }


def make_loader(data, transform, batch_size, shuffle, num_workers=0):
    """画像をテンソルに変換し、batch_size 枚ずつ束ねて取り出す（p92）"""
    return DataLoader(data.with_transform(partial(_apply, transform=transform)),
                      batch_size=batch_size, shuffle=shuffle, num_workers=num_workers)


train_loader = make_loader(train_data, train_transform, BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = make_loader(val_data, eval_transform, BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader = make_loader(test_data, eval_transform, BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

In [ ]:
batch = next(iter(train_loader))
print("入力 X の形 (枚数, チャネル, 高さ, 幅):", tuple(batch["image"].shape))
print("正解 Y の形 (枚数,):", tuple(batch["label"].shape), batch["label"][:8].tolist(), "...")
print()
print("1 epoch の更新の回数:", len(train_loader), f"（{len(train_data)} 枚 ÷ {BATCH_SIZE} 枚）")
print(f"{EPOCHS} epoch の更新の回数:", EPOCHS * len(train_loader))
viz.show_batch(batch, class_names)

## 4. モデル F とパラメータ θ（p73、p94）

### 4-1. ResNet-18 を作る（p67、p73）

`timm` というライブラリで、CNN の一つ ResNet-18 を作ります。`num_classes` を指定すると、分類ヘッドが37クラス用に付け替えられます（4-4）。

In [ ]:
def build_model(model_name, num_classes, pretrained=True):
    """F（ネットワークの形）を作り、θ（重み）に学習済みの値を入れる

    num_classes を指定すると、分類ヘッドが num_classes クラス用に付け替えられる（p94）。
    使えるモデルの名前は timm.list_models(pretrained=True) で調べられる。
    """
    return timm.create_model(model_name, pretrained=pretrained, num_classes=num_classes)


model = build_model(MODEL_NAME, num_classes, pretrained=PRETRAINED).to(device)

画像が層を通るごとに、特徴マップの形（チャネル, 高さ, 幅）がどう変わるかを見ます。

| 層の名前 | 役割 | スライド |
| --- | --- | --- |
| `conv1` | 畳み込み層（7×7 のカーネルが64枚） | p54、p65 |
| `bn1` | 値の範囲を整える層（スライドでは扱っていない） | |
| `act1` | ReLU | p68 |
| `maxpool` | Pooling 層。大きさを半分にする | p69 |
| `layer1`〜`layer4` | 畳み込み層などの積み重ね。だんだん小さく、チャネルは多くなる | p70 |
| `global_pool` | 特徴マップを、チャネルごとに1つの数にまとめる | p74 |
| `fc` | 全結合層。37クラスのスコアを出す | p74 |

`conv1`〜`layer4` が**特徴抽出器**、`global_pool` と `fc` が**分類ヘッド**です（p73）。

In [ ]:
viz.print_feature_shapes(model, device)

In [ ]:
# 中身をすべて表示する（長いので、興味があれば）
# print(model)

### 4-2. θ を数える（p58）

$\theta$ は、`model.parameters()` に入っている数（重みとバイアス）の全部です。学習では、これを少しずつ動かします。

In [ ]:
n_all = sum(p.numel() for p in model.parameters())
n_head = sum(p.numel() for p in model.get_classifier().parameters())
print(f"θ の数（全体）: {n_all:,}")
print(f"  特徴抽出器  : {n_all - n_head:,}")
print(f"  分類ヘッド  : {n_head:,}")
print()
print("最初の畳み込み層の重みの形 (枚数, 入力チャネル, 高さ, 幅):", tuple(model.conv1.weight.shape))
print("分類ヘッドの重みの形 (クラス数, 特徴の数):", tuple(model.fc.weight.shape))

### 4-3. 学習済みのカーネル（p65）

最初の畳み込み層のカーネル64枚です。人間が値を決めたのではなく、ImageNet で学習して求まったものです。境界や色に反応するものが見えます（真っ黒なものは、値がすべて0）。

`PRETRAINED = False` にすると、学習前のランダムな値になります。

In [ ]:
viz.show_kernels(model)

### 4-4. 分類ヘッドを付け替える（p94）

ImageNet で学習したモデルの分類ヘッドは1000クラス用です。今回は37クラス用に付け替えました。特徴抽出器の重みは引き継ぎ、新しい分類ヘッドの重みはランダムな値から始まります（転移学習）。

In [ ]:
original = timm.create_model(MODEL_NAME, pretrained=False)  # 付け替える前の形（重みは読み込まない）
print("元の分類ヘッド    :", original.get_classifier())
print("今回の分類ヘッド  :", model.get_classifier())
del original

### 4-5. 学習の前に予測してみる（p74）

分類ヘッドが出す37個のスコア（logits）を、softmax で合計1の確率に変換します。分類ヘッドがまだランダムなので、ほとんど当てずっぽうです（赤が正解のクラス）。

下の `predict` は、8. 推論でも使います。

In [ ]:
@torch.no_grad()
def predict(model, image, transform, device):
    """1枚の画像（PIL）を入力し、各クラスの確率を返す（推論。p17、p74）"""
    model.eval()
    x = transform(image.convert("RGB")).unsqueeze(0).to(device)  # 1枚だけの mini-batch にする
    return model(x).softmax(dim=1)[0].cpu()


probs = predict(model, sample["image"], eval_transform, device)
print("確率の合計:", round(probs.sum().item(), 4))
print("正解のクラスの確率:", round(probs[sample["label"]].item(), 4), f" （当てずっぽうなら 1/{num_classes} ≒ {1 / num_classes:.4f}）")
viz.show_probs(probs, class_names, true_label=sample["label"])

## 5. Loss と更新の方法（p75、p76）

### 5-1. Loss：交差エントロピー（p75）

分類では、正解のクラスの確率 $p$ を使って、Loss ＝ $-\log p$ とします。正解の確率が1に近いほど0に近く、0に近いほど大きくなります。

In [ ]:
criterion = nn.CrossEntropyLoss()

model.eval()
with torch.no_grad():
    logits = model(eval_transform(sample["image"]).unsqueeze(0).to(device))  # Ŷ = F(X; θ)
label = torch.tensor([sample["label"]], device=device)                        # Y
p = logits.softmax(dim=1)[0, sample["label"]]
print("criterion で計算した Loss:", round(criterion(logits, label).item(), 4))
print("−log(正解の確率)         :", round(-torch.log(p).item(), 4))

### 5-2. 更新の方法：AdamW（p25、p76）

勾配降下法（p25）では、$\theta$ を「勾配に応じて、Loss が小さくなる方向に少し動かす」ことを繰り返します。動かす幅を決めるのが学習率です（p26）。

ここでは、勾配降下法の改良版の一つ AdamW を使います。`model.parameters()`（$\theta$ の全部）を渡して、更新する対象にします。

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
optimizer

### 5-3. 1回だけ更新してみる（p16、p76）

3-6 で取り出した mini-batch 1つで、学習の繰り返しを1回だけ手で実行します。$\theta$ が少し動き、同じ mini-batch の Loss が下がることを確かめます。

In [ ]:
images = batch["image"].to(device)  # 入力 X
labels = batch["label"].to(device)  # 正解 Y


def batch_loss():
    model.eval()
    with torch.no_grad():
        return criterion(model(images), labels).item()


before_loss, before_w = batch_loss(), model.fc.weight[0, :3].tolist()

model.train()
logits = model(images)            # 予測を計算する：Ŷ = F(X; θ)
loss = criterion(logits, labels)  # Loss を計算する
optimizer.zero_grad()             # （前回の勾配を消す）
loss.backward()                   # 勾配を計算する（誤差逆伝播）
optimizer.step()                  # θ を更新する

print("θ の一部（分類ヘッドの重み3個）")
print("  更新前:", np.round(before_w, 4))
print("  更新後:", np.round(model.fc.weight[0, :3].tolist(), 4))
print(f"同じ mini-batch の Loss: {before_loss:.3f} → {batch_loss():.3f}")

## 6. ② 学習を進める（p90〜p92）

5-3 の1回の更新を、すべての mini-batch について行うと 1 epoch です。それを `EPOCHS` 回繰り返し、1周ごとに validation で正解率を測って、一番良かった時点の重みを残します（p90）。

`train_one_epoch` の中の4行が、スライドの学習の繰り返しです（p76）。

| 学習の繰り返し | コード |
| --- | --- |
| 予測を計算する $\hat{Y} = F(X; \theta)$ | `logits = model(images)` |
| Loss を計算する | `loss = criterion(logits, labels)` |
| 勾配を計算する（誤差逆伝播） | `loss.backward()` |
| $\theta$ を更新する | `optimizer.step()` |

`evaluate` は、$\theta$ を更新せずに Loss と正解率を測ります（validation と test で使う）。`model.train()` と `model.eval()` は、データ拡張とは別に、学習と評価で動きが変わる層（`bn1` など）を切り替えます。

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    """train を1周（1 epoch）学習する。mini-batch ごとに θ を1回更新する（p90〜92）"""
    model.train()
    total_loss, correct, count = 0.0, 0, 0
    for batch in loader:
        images = batch["image"].to(device)   # 入力 X
        labels = batch["label"].to(device)   # 正解 Y

        logits = model(images)               # 予測を計算する：Ŷ = F(X; θ)
        loss = criterion(logits, labels)     # Loss を計算する
        optimizer.zero_grad()                # （前回の勾配を消す）
        loss.backward()                      # 勾配を計算する（誤差逆伝播）
        optimizer.step()                     # θ を更新する

        total_loss += loss.item() * len(labels)
        correct += (logits.argmax(dim=1) == labels).sum().item()
        count += len(labels)
    return total_loss / count, correct / count


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    """θ を更新せずに、Loss と正解率を測る（validation、test。p86）"""
    model.eval()
    total_loss, count = 0.0, 0
    all_preds, all_labels = [], []
    for batch in loader:
        images = batch["image"].to(device)
        labels = batch["label"].to(device)
        logits = model(images)
        total_loss += criterion(logits, labels).item() * len(labels)
        count += len(labels)
        all_preds.append(logits.argmax(dim=1).cpu())
        all_labels.append(labels.cpu())
    preds, labels = torch.cat(all_preds), torch.cat(all_labels)
    return total_loss / count, (preds == labels).float().mean().item(), preds, labels

In [ ]:
history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
best_acc, best_state = 0.0, None
for epoch in range(1, EPOCHS + 1):
    start = time.time()
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc, _, _ = evaluate(model, val_loader, criterion, device)
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)
    print(f"epoch {epoch}/{EPOCHS}  train loss {train_loss:.3f} acc {train_acc:.3f}"
          f"  |  val loss {val_loss:.3f} acc {val_acc:.3f}  |  {time.time() - start:.0f}s")
    # validation の正解率が一番良い時点の重みを残す
    if val_acc > best_acc:
        best_acc, best_state = val_acc, copy.deepcopy(model.state_dict())

model.load_state_dict(best_state)
print(f"best val acc {best_acc:.3f}")

## 7. ③ 結果を確かめる（p97〜p99）

### 7-1. 学習曲線（p97）

train の Loss は下がり続けます。validation の Loss が途中から上がり始めたら、過学習のサインです。

In [ ]:
viz.plot_history(history)

### 7-2. test で測る（p98）

学習にも設定の判断にも使っていない test で、最後に1回だけ測ります。数字だけで終わらせず、どんな間違い方をしているかも見ます。

In [ ]:
test_loss, test_acc, preds, labels = evaluate(model, test_loader, criterion, device)
print(f"train      acc {history['train_acc'][-1]:.3f}  （{EPOCHS} epoch 目）")
print(f"validation acc {best_acc:.3f}  （最良）")
print(f"test       acc {test_acc:.3f}  （{len(labels)} 枚）")
print(f"参考：当てずっぽうなら 1/{num_classes} ≒ {1 / num_classes:.3f}")
print()
print("間違いが多かった組み合わせ（正解 -> 予測）")
viz.print_confusions(preds, labels, class_names)

In [ ]:
viz.show_mistakes(test_data, preds, labels, class_names)

### 7-3. 層ごとの特徴マップ（p71）

学習したモデルの中で、画像がどう変わっていくかを見ます。各層の特徴マップから、平均的に強く反応したものを6枚ずつ表示します。深い層ほど小さくなり、1マスが元の画像の広い範囲に対応します（p70）。

In [ ]:
cat = Image.open("assets/chelsea.png")
viz.show_feature_maps(model, cat, eval_transform, device)

### 7-4. 学習していないものを入れると（p99）

モデルは、学習した37クラスのどれかでしか答えられません。確率は合計が1になるように変換しただけなので、高くても正しいとは限りません。

In [ ]:
coffee = Image.open("assets/coffee.png")
probs = predict(model, coffee, eval_transform, device)
viz.show_prediction(coffee, probs, class_names)

## 8. 推論：好きな画像を入れる（p17）

学習が終われば、使い方は古典的手法と同じで、画像を入れると答えが出ます（$\theta$ は学習で求めた値に固定）。まず、test の画像と猫の画像で試します。

In [ ]:
probs = predict(model, test_data[0]["image"], eval_transform, device)
print("正解:", class_names[test_data[0]["label"]])
viz.show_prediction(test_data[0]["image"], probs, class_names)

probs = predict(model, cat, eval_transform, device)
viz.show_prediction(cat, probs, class_names)

次に、**自分で撮った画像**を入れてみましょう。`USE_UPLOAD = True` にして実行すると、アップロードのボタンが表示されます（Colab のみ）。犬や猫以外（タヌキ、ぬいぐるみ、自動車など）も試してください。

In [ ]:
USE_UPLOAD = False

if USE_UPLOAD:
    from google.colab import files

    for name in files.upload():
        viz.show_prediction(Image.open(name), predict(model, Image.open(name), eval_transform, device), class_names)

## 9. 同じ学習を train.py で動かす

研究では、GitHub で公開されたコードを動かし、読んで、直して使うことがよくあります。そうしたコードは、ノートブックではなく、役割ごとのファイルに分かれていて、コマンドで実行する形が多いです。

このフォルダの `.py` には、ここまでのセルと同じ処理が入っています。左の「ファイル」から開いて、見比べてみましょう。

| ファイル | 中身 | このノートブック |
| --- | --- | --- |
| `data.py` | `load_splits`、`build_transforms`、`make_loader` | 3. ① データを用意する |
| `models.py` | `build_model` | 4-1 |
| `engine.py` | `train_one_epoch`、`evaluate`、`predict` | 4-5、6 |
| `train.py` | 設定（`argparse`）、Loss、更新の方法、epoch の繰り返し、保存 | 2、5、6、7-2 |
| `predict.py` | 保存した重みで推論する | 8 |
| `viz.py` | 表示用 | （各節の図） |

学習の設定は、2. のセルの代わりに、コマンドの引数で渡します。使える引数は `--help` で分かります。

In [ ]:
!python train.py --help

In [ ]:
# GPU が無いときは枚数を減らす
options = f"--num-workers {NUM_WORKERS}" + ("" if device == "cuda" else " --max-train 600 --max-test 300")
!python train.py --epochs 3 {options}

In [ ]:
# 保存した重み（outputs/best.pt）で推論する
!python predict.py outputs/best.pt assets/chelsea.png assets/coffee.png

In [ ]:
# 保存した学習の記録（outputs/history.json）を表示する
saved = json.loads(Path("outputs/history.json").read_text(encoding="utf-8"))
print(f"best val acc {saved['best_val_acc']:.3f}  |  test acc {saved['test_acc']:.3f}")
viz.plot_history(saved)

### やってみよう

`--out` で保存先を変えると、結果を並べて比べられます。

1. 学習率を変える：`!python train.py --lr 0.1 --out outputs/lr0.1`（p26）
2. 学習済みの重みを使わない：`!python train.py --no-pretrained --out outputs/scratch`（p94）
3. モデルを変える：`!python train.py --model resnet50.a1_in1k --out outputs/resnet50`
4. **コードを直す**：`data.py` の `build_transforms` から `RandomHorizontalFlip` の行を消して実行し、結果を比べる（p88）

## 10. 課題（p102）

高い精度を競う課題ではありません。実験と考察の結果を PDF にまとめて提出してください。

**確認項目**

- [ ] 最後まで実行できた
- [ ] 学習に用意するもの（データ、モデル、Loss、更新の方法、設定）と学習の繰り返しの4行を、コードを指して説明できる
- [ ] train / validation / test の違いを説明できる
- [ ] 自分の画像で推論した
- [ ] 誤分類を1例以上取り上げ、原因を考えた（例：タヌキの画像を入れたら猫と判定された。耳の形が似ているからではないか）

**実験（1つ以上）**：2. の設定か、9. の引数を1つ変えて、結果を比べてください（学習率、epoch 数、学習済みの有無、モデル）。

**発展**：自分で集めた画像で分類器を作る。クラスごとにフォルダを分けて画像を入れ（例：`my_images/cat/`、`my_images/dog/`）、`IMAGE_DIR = "my_images"` にして実行します。`train.py` では `--image-dir my_images` です。